In [1]:
import duckdb
import pandas as pd

conn = duckdb.connect('../data/churn_database.duckdb')

In [2]:
query1 = """
SELECT 
    CASE WHEN churn = 1 THEN 'Churned (Lost Revenue)' ELSE 'Retained (Active MRR)' END as status,
    COUNT(*) as customer_count,
    ROUND(SUM(monthlycharges), 2) as total_mrr,
    ROUND(AVG(monthlycharges), 2) as arpu
FROM customers
GROUP BY churn;
"""
df_revenue = conn.execute(query1).df()
display(df_revenue)


,status,customer_count,total_mrr,arpu
0,Retained (Active MRR),900773,77980391.19,86.57
1,Churned (Lost Revenue),99227,8463309.54,85.29


In [3]:
query2 = """
SELECT 
    contract,
    COUNT(*) as churned_customers,
    ROUND(SUM(monthlycharges), 2) as lost_mrr,
    ROUND(SUM(totalcharges), 2) as lost_lifetime_value
FROM customers
WHERE churn = 1
GROUP BY contract
ORDER BY lost_mrr DESC;
"""
df_lost_by_contract = conn.execute(query2).df()
display(df_lost_by_contract)


,contract,churned_customers,lost_mrr,lost_lifetime_value
0,one_year,69651,5945851.51,1.222330e+08
1,two_year,24277,2070010.39,4.261252e+07
2,month_to_month,5299,447447.64,8.568071e+06


In [4]:
# 1. Export the full table
conn.execute("COPY customers TO '../data/reports/powerbi_raw_data.csv' (HEADER, DELIMITER ',')")

# 2. Export the risk profile CTE from yesterday as a dedicated table
query3 = """
COPY (
    SELECT 
        customer_id,
        CASE 
            WHEN contract = 'month_to_month' AND payment_method = 'electronic_check' THEN 'High Risk'
            WHEN contract = 'month_to_month' THEN 'Medium Risk'
            ELSE 'Low Risk'
        END as risk_profile
    FROM customers
) TO '../data/reports/powerbi_risk_profiles.csv' (HEADER, DELIMITER ',');
"""
conn.execute(query3)
print("Exported files to data/reports/ for Power BI!")


Exported files to data/reports/ for Power BI!


In [5]:
conn.close()